# Data Quality Report

This notebook combines Silver quarantine summaries and Silver/Gold quality metrics into reporting-ready datasets for monitoring and dashboards.

In [0]:
from pyspark.sql import functions as F

In [0]:
GOLD_QUALITY_BASE_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/data_quality"
)

SILVER_QUARANTINE_SUMMARY_PATH = (
    f"{GOLD_QUALITY_BASE_PATH}/silver_quarantine_summary"
)

REJECTION_REASON_SUMMARY_PATH = (
    f"{GOLD_QUALITY_BASE_PATH}/rejection_reason_summary"
)

DATA_QUALITY_METRICS_PATH = (
    f"{GOLD_QUALITY_BASE_PATH}/data_quality_metrics"
)

In [0]:
silver_quarantine_summary_df = (
    spark.read
    .format("delta")
    .load(SILVER_QUARANTINE_SUMMARY_PATH)
)

rejection_reason_summary_df = (
    spark.read
    .format("delta")
    .load(REJECTION_REASON_SUMMARY_PATH)
)

data_quality_metrics_df = (
    spark.read
    .format("delta")
    .load(DATA_QUALITY_METRICS_PATH)
)

print("Quarantine summary rows:", silver_quarantine_summary_df.count())
print("Rejection reason rows:", rejection_reason_summary_df.count())
print("Quality metric rows:", data_quality_metrics_df.count())

In [0]:
quality_overview_df = (
    data_quality_metrics_df
    .groupBy("layer")
    .agg(
        F.count("*").alias("table_count"),
        F.sum(
            F.when(F.col("quality_status") == "PASS", 1).otherwise(0)
        ).alias("passed_table_count"),
        F.sum(
            F.when(F.col("quality_status") != "PASS", 1).otherwise(0)
        ).alias("review_table_count"),
        F.sum("row_count").alias("total_row_count"),
        F.sum("null_business_key_count").alias(
            "total_null_business_key_count"
        ),
        F.sum("duplicate_grain_count").alias(
            "total_duplicate_grain_count"
        ),
    )
    .withColumn(
        "pass_rate",
        F.round(
            F.col("passed_table_count") / F.col("table_count") * 100,
            2,
        ),
    )
    .withColumn("_quality_reported_at", F.current_timestamp())
    .orderBy("layer")
)

display(quality_overview_df)

In [0]:
quarantine_overview_df = (
    silver_quarantine_summary_df
    .agg(
        F.count("*").alias("dataset_count"),
        F.sum("rejected_row_count").alias("total_rejected_row_count"),
        F.sum(
            F.when(F.col("rejected_row_count") > 0, 1).otherwise(0)
        ).alias("datasets_with_rejections"),
        F.sum(
            F.when(F.col("rejected_row_count") == 0, 1).otherwise(0)
        ).alias("datasets_without_rejections"),
    )
    .withColumn("_quality_reported_at", F.current_timestamp())
)

display(quarantine_overview_df)

In [0]:
QUALITY_OVERVIEW_PATH = (
    f"{GOLD_QUALITY_BASE_PATH}/quality_overview"
)

QUARANTINE_OVERVIEW_PATH = (
    f"{GOLD_QUALITY_BASE_PATH}/quarantine_overview"
)

In [0]:
(
    quality_overview_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUALITY_OVERVIEW_PATH)
)

(
    quarantine_overview_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_OVERVIEW_PATH)
)

In [0]:
saved_quality_overview_df = (
    spark.read
    .format("delta")
    .load(QUALITY_OVERVIEW_PATH)
)

saved_quarantine_overview_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_OVERVIEW_PATH)
)

print("Quality overview rows:", saved_quality_overview_df.count())
print("Quarantine overview rows:", saved_quarantine_overview_df.count())

display(saved_quality_overview_df)
display(saved_quarantine_overview_df)